In [62]:
import os
import requests
import zipfile
import io
import pandas as pd
import numpy as np
import IPython.display as ipd
from sklearn.preprocessing import StandardScaler

In [63]:
#settings
url = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"
zip_name = "fma_metadata.zip"

#creating a folder
os.makedirs('cache', exist_ok=True)

#cheking 
if not os.path.exists(os.path.join('cache', 'fma_metadata/tracks.csv')):
    
    # downloading file
    response = requests.get(url)
    
    if response.status_code == 200:
        #Unzipping
        with zipfile.ZipFile(io.BytesIO(response.content)) as f:
            f.extractall('cache')

In [64]:
# downloaing medium dataset
df_tracks = pd.read_csv('cache/fma_metadata/tracks.csv',index_col=0, header=[0, 1]) 
tracks_medium = df_tracks[df_tracks['set', 'subset'] == 'medium']

# leaving only 8 
target_genres = ['Hip-Hop', 'Pop', 'Folk', 'Rock', 'Experimental', 'International', 'Electronic', 'Instrumental']
tracks = tracks_medium[tracks_medium['track', 'genre_top'].isin(target_genres)]

df_features = pd.read_csv('cache/fma_metadata/features.csv',index_col=0, header=[0, 1, 2]) 

In [65]:
common_ids = tracks.index.intersection(df_features.index)

X= df_features.loc[common_ids]  
y= tracks.loc[common_ids, ('track', 'genre_top')]

## Split

In [66]:
#splittingdata
split_info = tracks.loc[common_ids, ('set', 'split')]

X_train = X[split_info == 'training']
y_train = y[split_info == 'training']

X_val = X[split_info == 'validation']
y_val = y[split_info == 'validation']

X_test = X[split_info == 'test']
y_test = y[split_info == 'test']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (11912, 518), Val: (1495, 518), Test: (1535, 518)


In [67]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)


X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [68]:
def achlioptas_matrix(d, l):
    vals = [np.sqrt(3), 0, -np.sqrt(3)]
    probs = [1/6, 2/3, 1/6]
    return np.random.choice(vals, size=(d, l), p=probs)

In [69]:
l = 10  
n = 15   
d = X_train_scaled.shape[1] 
    
lsh_tables = []
r_matrices = []

for i in range(n):
    R = achlioptas_matrix(d, l)
    r_matrices.append(R)
    table = {} 
    
    projections = np.dot(X_train_scaled, R)
    hashes = (projections >= 0).astype(int)
    
    for idx, h in enumerate(hashes):
        hash_tuple = tuple(h)
        if hash_tuple not in table:
            table[hash_tuple] = []
        table[hash_tuple].append(idx) 
    
    lsh_tables.append(table)

In [80]:
def predict_genre(X_val_scaled, k, lsh_tables, r_matrices, X_train_scaled, y_train):

    
    candidate_indices = set()
    
    # Collecting all candidates fom all n tabels
    for i in range(len(lsh_tables)):
        # Calculating hash
        projection = np.dot(X_val_scaled,r_matrices[i])
        h = tuple((projection >= 0).astype(int))
        
        
        if h in lsh_tables[i]:
            candidate_indices.update(lsh_tables[i][h])
    

    if not candidate_indices:
        return "Unknown"
    
    
    candidate_list = list(candidate_indices)
    

    distances = []
    for idx in candidate_list:
        dist = np.linalg.norm(X_val_scaled - X_train_scaled[idx])
        distances.append(dist)
    
 
    sorted_neighbor_indices = np.argsort(distances)[:k]
    
    top_k_indices = [candidate_list[i] for i in sorted_neighbor_indices]

    neighbor_genres = y_train.iloc[top_k_indices].values
    
    vote_counts = {}
    for genre in neighbor_genres:
        vote_counts[genre] = vote_counts.get(genre, 0) + 1
     most_common_genre = max(vote_counts, key=vote_counts.get)
    
    return most_common_genre

In [83]:
k = 10
predictions = []

for track in X_val_scaled:
    res = predict_genre(track, k, lsh_tables, r_matrices, X_train_scaled, y_train)
    predictions.append(res)

accuracy = np.mean(predictions == y_val.values)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 66.76%
